# Network traffic classification — UNSW-NB15
**Goal:** classify each network flow as normal (0) or malicious (1).

Run cells from top to bottom in Google Colab (CPU runtime is sufficient). Upload both supplied Excel files when prompted. CSV files are also supported. The full datasets are used, not a training sample.

**Order:** data audit → Matplotlib/Seaborn visualization → validation split → Logistic Regression → Random Forest → XGBoost → LightGBM → final comparison.

**Primary metric:** binary F1 for malicious traffic: `2 × precision × recall / (precision + recall)`. Macro F1, precision, recall, accuracy and average precision are supporting metrics. A fixed 0.5 probability threshold makes this a basic, reproducible comparison. Select the model on validation F1, never on test F1.

The uploaded workbooks were checked to contain 175,341 training rows, 82,332 test rows and the same 45-column schema. Model scores are calculated when you run this notebook; none are fabricated. Test data is not used for fitting or tuning. Initial test visualizations are limited to structural quality checks; detailed test outcomes appear only after model selection.

In [ ]:
%pip -q install "scikit-learn==1.6.1" "xgboost==2.1.4" "lightgbm==4.6.0" openpyxl seaborn joblib

In [ ]:
import os, time, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.metrics import (f1_score, precision_score, recall_score, accuracy_score,
    average_precision_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, precision_recall_curve)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib
SEED = 42
N_JOBS = min(4, os.cpu_count() or 2)
THRESHOLD = 0.5
OUT = Path('results'); OUT.mkdir(exist_ok=True)
sns.set_theme(style='whitegrid', palette='Set2')
pd.set_option('display.max_columns', 50)
def show_save(name):
    plt.tight_layout()
    plt.savefig(OUT / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.show()


## 1. Upload and inspect the data
Upload the training and testing files together. Excel loading can take several minutes. Keep `attack_cat` for descriptive analysis only; it must never enter the model.

In [ ]:
from google.colab import files
uploaded = files.upload()
names = list(uploaded)
def choose_file(keyword):
    matches = [p for p in names if keyword in p.lower() and Path(p).suffix.lower() in ['.xlsx', '.csv']]
    if len(matches) != 1:
        raise ValueError(f'Upload exactly one {keyword} file. Found: {matches}')
    return matches[0]
def read_data(path):
    return pd.read_excel(path, engine='openpyxl') if path.lower().endswith('.xlsx') else pd.read_csv(path)
train_df = read_data(choose_file('training'))
test_df = read_data(choose_file('testing'))
del uploaded
for df in [train_df, test_df]:
    df.columns = df.columns.str.strip()
    assert df.columns.is_unique
    assert {'id','label','attack_cat'}.issubset(df.columns)
    assert df['label'].notna().all() and set(df['label'].unique()) == {0,1}
    df['label'] = df['label'].astype(int)
assert set(train_df.columns) == set(test_df.columns), 'Train/test schemas differ'
test_df = test_df[train_df.columns]
display(pd.DataFrame({'Dataset':['Train','Test'], 'Rows':[len(train_df),len(test_df)],
                      'Columns':[train_df.shape[1],test_df.shape[1]]}))
display(train_df.head())
display(train_df.describe(include='all').T)


In [ ]:
FEATURES = [c for c in train_df if c not in ['id','label','attack_cat']]
CAT_COLS = train_df[FEATURES].select_dtypes(include=['object','string','category']).columns.tolist()
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]
def clean_inputs(df):
    x = df[FEATURES].copy()
    for c in CAT_COLS:
        x[c] = x[c].fillna('Unknown').astype(str).str.strip()
    for c in NUM_COLS:
        x[c] = pd.to_numeric(x[c], errors='coerce')
    return x.replace([np.inf, -np.inf], np.nan)
X = clean_inputs(train_df); y = train_df['label'].copy()
X_test = clean_inputs(test_df); y_test = test_df['label'].copy()
# Identical predictor rows stay in the same validation fold.
groups = pd.util.hash_pandas_object(X, index=False)
test_groups = pd.util.hash_pandas_object(X_test, index=False)
overlap_mask = test_groups.isin(set(groups))
audit = pd.DataFrame({
    'Train': [len(X), len(FEATURES), int(X.isna().sum().sum()), int(X.duplicated().sum())],
    'Test': [len(X_test), len(FEATURES), int(X_test.isna().sum().sum()), int(X_test.duplicated().sum())]},
    index=['Rows','Predictor columns','Missing predictor cells','Repeated predictor rows'])
display(audit)
print('Categorical columns:', CAT_COLS)
print('Test rows with identical predictors in training:', int(overlap_mask.sum()))
conflicting = pd.DataFrame({'group':groups, 'label':y}).groupby('group')['label'].nunique().gt(1).sum()
print('Training feature groups with conflicting labels:', conflicting)
print('The service value "-" is retained as a category, not assumed to be a missing number.')
audit.to_csv(OUT / 'data_audit.csv')


## 2. Visualize the full training dataset
These plots aggregate all training records. Every numeric feature is included in histogram and boxplot grids; every categorical feature is plotted. Log scales help display skewed distributions. Test distributions are reserved for the final evaluation to reduce test-informed model choices.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,4))
sns.countplot(data=train_df, x='label', ax=axes[0])
axes[0].set_xticks([0,1], ['Normal (0)','Malicious (1)'])
axes[0].set_title('Training class balance')
for container in axes[0].containers: axes[0].bar_label(container)
sns.countplot(data=train_df, y='attack_cat', order=train_df['attack_cat'].value_counts().index, ax=axes[1])
axes[1].set_title('Training attack categories — analysis only')
show_save('training_class_balance')
display(train_df['label'].value_counts().rename('Count').to_frame().assign(
    Percentage=lambda t: 100*t['Count']/len(train_df)))
missing = pd.DataFrame({'Train':X.isna().mean()*100, 'Test':X_test.isna().mean()*100})
fig, ax = plt.subplots(figsize=(10,3))
sns.heatmap(missing.T, cmap='YlOrRd', ax=ax, cbar_kws={'label':'Missing %'})
ax.set_title('Missing predictor values in both files')
show_save('missing_values')


In [ ]:
for start in range(0, len(NUM_COLS), 12):
    columns = NUM_COLS[start:start+12]
    fig, axes = plt.subplots(3,4,figsize=(16,10))
    for ax,c in zip(axes.flat,columns):
        sns.histplot(X[c], bins=40, ax=ax, color='steelblue')
        ax.set_yscale('symlog'); ax.set_title(c)
    for ax in list(axes.flat)[len(columns):]: ax.set_visible(False)
    fig.suptitle('Numeric distributions — all training rows (log count axis)', y=1.02)
    show_save(f'histograms_{start}')
for start in range(0, len(NUM_COLS), 12):
    columns = NUM_COLS[start:start+12]
    fig, axes = plt.subplots(3,4,figsize=(16,10))
    for ax,c in zip(axes.flat,columns):
        sns.boxplot(x=y, y=X[c], ax=ax, showfliers=False)
        ax.set_yscale('symlog'); ax.set_title(c); ax.set_xlabel('0 normal | 1 malicious')
    for ax in list(axes.flat)[len(columns):]: ax.set_visible(False)
    fig.suptitle('Features by class — log value axis; outlier points hidden, not removed', y=1.02)
    show_save(f'boxplots_{start}')


In [ ]:
for c in CAT_COLS:
    counts = X[c].value_counts()
    display(counts.rename('Count').to_frame())
    plot_category = X[c].where(X[c].isin(counts.head(15).index), 'Other categories')
    fig, ax = plt.subplots(figsize=(11,5))
    sns.countplot(y=plot_category, hue=y, ax=ax, order=plot_category.value_counts().index)
    ax.set_title(f'{c} by class — top 15 categories plus all remaining categories')
    ax.legend(title='Label'); show_save(f'categorical_{c}')
plt.figure(figsize=(16,13))
sns.heatmap(X[NUM_COLS].corr(method='spearman'), cmap='coolwarm', center=0, vmin=-1,vmax=1)
plt.title('Training numeric feature correlations (Spearman)')
show_save('correlation_heatmap')
print('A correlation is an association, not evidence that a feature causes an attack.')


## 3. Split and preprocess
Use one approximately 80/20 stratified group split for every model. Duplicate predictor rows cannot cross training/validation boundaries. Keep the supplied test set separate. Imputation, encoding and scaling are learned within each pipeline. Scaling is used for Logistic Regression; trees do not need it. No oversampling or class weighting is used in this initial baseline.

In [ ]:
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
fit_idx, val_idx = next(splitter.split(X,y,groups))
X_fit, X_val = X.iloc[fit_idx], X.iloc[val_idx]
y_fit, y_val = y.iloc[fit_idx], y.iloc[val_idx]
assert set(groups.iloc[fit_idx]).isdisjoint(set(groups.iloc[val_idx]))
display(pd.DataFrame({'Rows':[len(y_fit),len(y_val)],
    'Malicious fraction':[y_fit.mean(),y_val.mean()]}, index=['Fit','Validation']))
def make_preprocessor(scale=False):
    num_steps = [('imputer',SimpleImputer(strategy='median',keep_empty_features=True))]
    if scale: num_steps.append(('scale',StandardScaler()))
    return ColumnTransformer([
        ('num', Pipeline(num_steps), NUM_COLS),
        ('cat', OneHotEncoder(handle_unknown='ignore',sparse_output=True,dtype=np.float32), CAT_COLS)
    ], sparse_threshold=1.0)
def make_pipeline(model, scale=False):
    return Pipeline([('preprocess',make_preprocessor(scale)), ('model',model)])
def metric_row(y_true,p):
    pred = (p >= THRESHOLD).astype(int)
    return {'F1':f1_score(y_true,pred,zero_division=0),
            'Macro F1':f1_score(y_true,pred,average='macro',zero_division=0),
            'Precision':precision_score(y_true,pred,zero_division=0),
            'Recall':recall_score(y_true,pred,zero_division=0),
            'Accuracy':accuracy_score(y_true,pred),
            'Average precision':average_precision_score(y_true,p)}
models = {}; validation_rows=[]
def train_validate(name, pipeline):
    start=time.perf_counter(); pipeline.fit(X_fit,y_fit)
    elapsed=time.perf_counter()-start
    train_metrics=metric_row(y_fit,pipeline.predict_proba(X_fit)[:,1])
    val_metrics=metric_row(y_val,pipeline.predict_proba(X_val)[:,1])
    validation_rows.append({'Model':name,'Train F1':train_metrics['F1'],
                            **val_metrics,'Fit seconds':elapsed})
    models[name]=pipeline
    display(pd.DataFrame([validation_rows[-1]]).round(4))


## 4. Logistic Regression — baseline

In [ ]:
train_validate('Logistic Regression', make_pipeline(
    LogisticRegression(max_iter=3000, solver='lbfgs', C=1.0, random_state=SEED), scale=True))

## 5. Random Forest

In [ ]:
train_validate('Random Forest', make_pipeline(RandomForestClassifier(
    n_estimators=250,max_depth=20,min_samples_leaf=2,n_jobs=N_JOBS,random_state=SEED)))

## 6. XGBoost

In [ ]:
train_validate('XGBoost', make_pipeline(XGBClassifier(
    n_estimators=300,max_depth=6,learning_rate=0.08,subsample=0.8,colsample_bytree=0.8,
    objective='binary:logistic',eval_metric='logloss',tree_method='hist',
    n_jobs=N_JOBS,random_state=SEED)))

## 7. LightGBM

In [ ]:
train_validate('LightGBM', make_pipeline(LGBMClassifier(
    n_estimators=300,learning_rate=0.08,num_leaves=31,subsample=0.8,subsample_freq=1,
    colsample_bytree=0.8,verbosity=-1,n_jobs=N_JOBS,random_state=SEED)))

## 8. Select on validation F1 and lock the experiment
These are specified baseline configurations, not exhaustive hyperparameter searches. If you tune anything, do it here using validation data, before running the test section. Convergence warnings from Logistic Regression should be resolved before comparing its score.

In [ ]:
validation = pd.DataFrame(validation_rows).set_index('Model')
winner = validation['F1'].idxmax()
print('Selected by validation malicious-class F1:',winner)
display(validation.sort_values('F1',ascending=False).round(4))
validation.to_csv(OUT/'validation_metrics.csv')
validation[['Train F1','F1']].rename(columns={'F1':'Validation F1'}).plot.bar(figsize=(10,5),ylim=(0,1))
plt.ylabel('Malicious-class F1'); plt.xticks(rotation=15); show_save('training_validation_f1')


## 9. Refit on all training records and evaluate the separate test set
All four locked configurations are refitted on the full training set. Test scores are reported for comparison, but the model selection made above is unchanged. Repeatedly adjusting models after viewing test results would invalidate this held-out evaluation. Identical train/test feature rows can inflate scores; a non-overlapping test subset is reported separately when available.

In [ ]:
final_models={}; test_probabilities={}; test_rows=[]; unique_rows=[]
for name, pipeline in models.items():
    fitted=clone(pipeline)
    start=time.perf_counter(); fitted.fit(X,y); elapsed=time.perf_counter()-start
    p=fitted.predict_proba(X_test)[:,1]
    final_models[name]=fitted; test_probabilities[name]=p
    test_rows.append({'Model':name, **metric_row(y_test,p),'Refit seconds':elapsed})
    if (~overlap_mask).any() and y_test.loc[~overlap_mask].nunique()==2:
        unique_rows.append({'Model':name,**metric_row(y_test.loc[~overlap_mask],p[~overlap_mask.to_numpy()])})
    print(name, 'completed')
test_results=pd.DataFrame(test_rows).set_index('Model')
display(test_results.round(4)); test_results.to_csv(OUT/'test_metrics.csv')
if unique_rows:
    print('Sensitivity check: test records absent from training by exact predictor match')
    unique_results=pd.DataFrame(unique_rows).set_index('Model')
    display(unique_results.round(4)); unique_results.to_csv(OUT/'nonoverlapping_test_metrics.csv')
print('Chosen model remains:',winner)


## 10. Final metric visualizations

In [ ]:
comparison = pd.DataFrame({'Train F1':validation['Train F1'],
    'Validation F1':validation['F1'],'Test F1 (full-data refit)':test_results['F1']})
ax=comparison.plot.bar(figsize=(12,5),ylim=(0,1.06))
for container in ax.containers: ax.bar_label(container,fmt='%.3f',fontsize=8)
plt.ylabel('Malicious-class F1'); plt.title('Four-model F1 comparison (threshold 0.5)')
plt.xticks(rotation=15); show_save('final_f1_comparison')
plt.figure(figsize=(9,4))
sns.heatmap(test_results[['F1','Macro F1','Precision','Recall','Accuracy','Average precision']],
            annot=True,fmt='.3f',cmap='YlGnBu',vmin=0,vmax=1)
plt.title('Final test metrics'); show_save('test_metric_heatmap')
fig,axes=plt.subplots(2,2,figsize=(10,8))
for ax,(name,p) in zip(axes.flat,test_probabilities.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test,p>=THRESHOLD,labels=[0,1]),
                           display_labels=['Normal','Malicious']).plot(ax=ax,colorbar=False,cmap='Blues')
    ax.set_title(name)
show_save('test_confusion_matrices')
plt.figure(figsize=(9,6))
for name,p in test_probabilities.items():
    precision, recall, _=precision_recall_curve(y_test,p)
    plt.plot(recall,precision,label=f'{name} (AP={average_precision_score(y_test,p):.3f})')
plt.axhline(y_test.mean(),linestyle='--',color='grey',label='Malicious prevalence')
plt.xlabel('Recall');plt.ylabel('Precision');plt.title('Test precision–recall curves');plt.legend()
show_save('test_precision_recall')
print(classification_report(y_test,test_probabilities[winner]>=THRESHOLD,
                            target_names=['Normal','Malicious'],zero_division=0))


## 11. Save and download
The saved pipeline includes preprocessing, so inference accepts the original 42 predictor columns. Only load joblib files you trust. The exported ZIP contains plots, metric tables, test predictions and the validation-selected model.

In [ ]:
joblib.dump(final_models[winner],OUT/'selected_pipeline.joblib')
predictions=pd.DataFrame({'id':test_df['id'],'actual_label':y_test})
for name,p in test_probabilities.items():
    predictions[name+'_probability']=p
    predictions[name+'_prediction']=(p>=THRESHOLD).astype(int)
predictions.to_csv(OUT/'test_predictions.csv',index=False)
with open(OUT/'experiment.json','w') as f:
    json.dump({'selected_model':winner,'selection_metric':'validation binary F1, positive label 1',
               'threshold':THRESHOLD,'seed':SEED,'features':FEATURES,
               'train_rows':len(X),'test_rows':len(X_test),
               'test_rows_overlapping_train':int(overlap_mask.sum())},f,indent=2)
import shutil
shutil.make_archive('unsw_nb15_results','zip',OUT)
files.download('unsw_nb15_results.zip')


## Interpretation and references
- Higher malicious-class F1 balances precision and recall; it does not measure all security costs. Review false positives and missed attacks too.
- Training F1 is in-sample; validation F1 comes from the development split; test F1 comes from models refitted on the full training set. Different distributions and training sizes can change these scores.
- This is an offline benchmark, not evidence of reliable detection on a live network or unseen attack families.
- [scikit-learn F1 definition](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)
- [XGBoost histogram tree method](https://xgboost.readthedocs.io/en/stable/treemethod.html)
- [LightGBM classifier](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.LGBMClassifier.html)
